In [1]:
# 11_variance_lodo — Cell 1: 5-seed CI + leave-one-domain-out (TF-IDF+LR), c2020 & c2025
import os, numpy as np, pandas as pd
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

P = "/mnt/g/banglafake-detection/data/processed/v2"
R = "/mnt/g/banglafake-detection/reports/v2"; os.makedirs(R, exist_ok=True)
SEEDS = [0, 1, 2, 3, 4]
MIN_N, MAX_FOLDS = 30, 20     # LODO: কমপক্ষে ৩০ রো-র domain, সর্বোচ্চ ২০টি fold (বড় domain আগে)

CORP = {n: pd.read_csv(f"{P}/{n}.csv") for n in ("c2020", "c2025")}
for df in CORP.values():
    df["text"] = df["text"].fillna("").str[:1500]
    df["domain"] = df["domain"].fillna("unk")
    df["label"] = df["label"].astype(int)

def metrics(y, s):
    p = (s > 0).astype(int)
    return dict(bacc=balanced_accuracy_score(y, p),
                f1=f1_score(y, p, average="macro"),
                auc=roc_auc_score(y, s))

def fit_scores(tr, te):
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=200000,
                        sublinear_tf=True, min_df=3)
    clf = LogisticRegression(max_iter=300, class_weight="balanced", C=2.0)
    clf.fit(v.fit_transform(tr["text"]), tr["label"])
    return clf.decision_function(v.transform(te["text"]))

# ---------- STAGE 1: ৫ seed, random split vs domain-held-out ----------
rows = []
for name, df in CORP.items():
    for seed in SEEDS:
        tr, te = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=seed)
        rows.append(dict(corpus=name, split="random_split", seed=seed, n_test=len(te),
                         **metrics(te["label"].values, fit_scores(tr, te))))

        gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
        tr_i, te_i = next(gss.split(df, groups=df["domain"]))
        tr, te = df.iloc[tr_i], df.iloc[te_i]
        rows.append(dict(corpus=name, split="domain_heldout", seed=seed, n_test=len(te),
                         **metrics(te["label"].values, fit_scores(tr, te))))
        print(name, "seed", seed, "done")

seed_df = pd.DataFrame(rows)
seed_df.to_csv(f"{R}/seed_runs_tfidf.csv", index=False)

def ci95(x):
    x = np.asarray(x); n = len(x)
    return stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)

summ = []
for (c, sp), g in seed_df.groupby(["corpus", "split"]):
    d = dict(corpus=c, split=sp, n_seeds=len(g))
    for m in ("bacc", "f1", "auc"):
        d[f"{m}_mean"] = round(g[m].mean(), 3)
        d[f"{m}_std"] = round(g[m].std(ddof=1), 3)
        d[f"{m}_ci95"] = round(ci95(g[m]), 3)
    summ.append(d)
summ = pd.DataFrame(summ)
summ.to_csv(f"{R}/seed_summary_tfidf.csv", index=False)
print("\n=== 5-seed summary ===")
print(summ.to_string(index=False))

# ---------- STAGE 2: leave-one-domain-out ----------
lodo_rows, pooled = [], {}
for name, df in CORP.items():
    counts = df["domain"].value_counts()
    elig = [d for d, c in counts.items()
            if d != "unk" and c >= MIN_N and df.loc[df["domain"] == d, "label"].nunique() == 2][:MAX_FOLDS]
    print(f"\n{name}: {len(counts)} domains total, {len(elig)} eligible folds")
    ys, ss = [], []
    for d in elig:
        te = df[df["domain"] == d]; tr = df[df["domain"] != d]
        s = fit_scores(tr, te); y = te["label"].values
        lodo_rows.append(dict(corpus=name, held_out_domain=d, n_test=len(te),
                              fake_frac=round(y.mean(), 3),
                              **{k: round(v, 3) for k, v in metrics(y, s).items()}))
        ys.append(y); ss.append(s)
        print("  ", d, len(te), "done")
    pooled[name] = metrics(np.concatenate(ys), np.concatenate(ss))

lodo = pd.DataFrame(lodo_rows)
lodo.to_csv(f"{R}/lodo_tfidf.csv", index=False)
print("\n=== LODO per-domain ===")
print(lodo.to_string(index=False))
print("\n=== LODO macro-mean (domain গড়) ===")
print(lodo.groupby("corpus")[["bacc", "f1", "auc"]].agg(["mean", "std"]).round(3))
print("\n=== LODO pooled ===")
for k, v in pooled.items():
    print(k, {m: round(x, 3) for m, x in v.items()})

c2020 seed 0 done
c2020 seed 1 done
c2020 seed 2 done
c2020 seed 3 done
c2020 seed 4 done
c2025 seed 0 done
c2025 seed 1 done
c2025 seed 2 done
c2025 seed 3 done
c2025 seed 4 done

=== 5-seed summary ===
corpus          split  n_seeds  bacc_mean  bacc_std  bacc_ci95  f1_mean  f1_std  f1_ci95  auc_mean  auc_std  auc_ci95
 c2020 domain_heldout        5      0.858     0.087      0.108    0.793   0.059    0.074     0.980    0.009     0.012
 c2020   random_split        5      0.943     0.009      0.011    0.901   0.011    0.013     0.991    0.003     0.003
 c2025 domain_heldout        5      0.686     0.093      0.116    0.587   0.031    0.038     0.773    0.061     0.075
 c2025   random_split        5      0.956     0.004      0.005    0.956   0.004    0.005     0.991    0.001     0.001

c2020: 93 domains total, 13 eligible folds
   kalerkantho.com 4430 done
   jagonews24.com 4423 done
   banglanews24.com 4037 done
   banglatribune.com 3692 done
   jugantor.com 2835 done
   dhakatimes24.co

In [4]:
import os, pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

P = "/mnt/g/banglafake-detection/data/processed/v2"
CORP = {n: pd.read_csv(f"{P}/{n}.csv") for n in ("c2020", "c2025")}
for df in CORP.values():
    df["text"] = df["text"].fillna("").str[:1500]
    df["domain"] = df["domain"].fillna("unk")
    df["label"] = df["label"].astype(int)

In [5]:
# 11 — Cell 2: domain-label confound + domain-only baseline + GroupKFold(5)
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GroupKFold

# (a) domain purity
for name, df in CORP.items():
    g = df.groupby("domain")["label"].agg(["size", "mean"])
    g["minor"] = np.minimum(g["mean"], 1 - g["mean"])
    pure = g[g["minor"] < 0.05]
    print(f"{name}: {len(g)} domains | {len(pure)} domains >95% এক ক্লাস, মোট রো-র {pure['size'].sum()/len(df):.1%}")
    print("  label mean:", round(df["label"].mean(), 3))

# (b) domain-only baseline (টেক্সট ছাড়া, শুধু domain)
rows = []
for name, df in CORP.items():
    for seed in SEEDS:
        tr, te = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=seed)
        enc = OneHotEncoder(handle_unknown="ignore")
        clf = LogisticRegression(max_iter=300, class_weight="balanced")
        clf.fit(enc.fit_transform(tr[["domain"]]), tr["label"])
        s = clf.decision_function(enc.transform(te[["domain"]]))
        rows.append(dict(corpus=name, seed=seed, **metrics(te["label"].values, s)))
dom = pd.DataFrame(rows)
dom.to_csv(f"{R}/domain_only_baseline.csv", index=False)
print("\n=== domain-only baseline (random split) ===")
print(dom.groupby("corpus")[["bacc", "f1", "auc"]].agg(["mean", "std"]).round(3))

# (c) GroupKFold(5) — প্রতি fold-এ একাধিক domain, দুই ক্লাসই থাকে
res = []
for name, df in CORP.items():
    ys, ss = [], []
    for fold, (tr_i, te_i) in enumerate(GroupKFold(n_splits=5).split(df, groups=df["domain"])):
        tr, te = df.iloc[tr_i], df.iloc[te_i]
        s = fit_scores(tr, te); y = te["label"].values
        ys.append(y); ss.append(s)
        if len(set(y)) == 2:
            res.append(dict(corpus=name, fold=fold, n_test=len(te), n_class0=int((y == 0).sum()),
                            **{m: round(v, 3) for m, v in metrics(y, s).items()}))
    print(name, "pooled:", {m: round(v, 3) for m, v in metrics(np.concatenate(ys), np.concatenate(ss)).items()})
gk = pd.DataFrame(res)
gk.to_csv(f"{R}/groupkfold_tfidf.csv", index=False)
print("\n=== GroupKFold per-fold ===")
print(gk.to_string(index=False))

c2020: 93 domains | 93 domains >95% এক ক্লাস, মোট রো-র 100.0%
  label mean: 0.974
c2025: 26 domains | 26 domains >95% এক ক্লাস, মোট রো-র 100.0%
  label mean: 0.501


NameError: name 'SEEDS' is not defined

In [3]:
# 11 — Cell 3: class-purity of domains + per-class recall (GroupKFold pooled)
for name, df in CORP.items():
    g = df.groupby("domain")["label"].agg(["size", "mean"])
    maj = (g["mean"] >= 0.5).astype(int)
    print(f"\n{name} | label-1 domains: {(maj==1).sum()} ({g.loc[maj==1,'size'].sum()} rows) "
          f"| label-0 domains: {(maj==0).sum()} ({g.loc[maj==0,'size'].sum()} rows)")
    print("  label-0 top:", g[maj==0].sort_values("size", ascending=False).head(5).index.tolist())
    print("  label-1 top:", g[maj==1].sort_values("size", ascending=False).head(5).index.tolist())
    ys, ss = [], []
    for tr_i, te_i in GroupKFold(n_splits=5).split(df, groups=df["domain"]):
        tr, te = df.iloc[tr_i], df.iloc[te_i]
        ss.append(fit_scores(tr, te)); ys.append(te["label"].values)
    y, s = np.concatenate(ys), np.concatenate(ss); p = (s > 0).astype(int)
    print("  recall label0:", round(((p==0)&(y==0)).sum()/(y==0).sum(), 3),
          "| recall label1:", round(((p==1)&(y==1)).sum()/(y==1).sum(), 3),
          "| AUC:", round(roc_auc_score(y, s), 3))


c2020 | label-1 domains: 22 (48545 rows) | label-0 domains: 71 (1276 rows)
  label-0 top: ['channeldhaka.news', 'earki.com', 'motikontho.wordpress.com', 'bengalbeats.com', 'bengaliviralnews.com']
  label-1 top: ['kalerkantho.com', 'jagonews24.com', 'banglanews24.com', 'banglatribune.com', 'jugantor.com']
  recall label0: 0.621 | recall label1: 0.989 | AUC: 0.971

c2025 | label-1 domains: 12 (2020 rows) | label-0 domains: 14 (1959 rows)
  label-0 top: ['Facebook Page', 'Earki.com', 'Somoy News', 'Deshrupantor', 'Dhaka Post']
  label-1 top: ['Prothomalo', 'Samakal News', 'Ittefaq', 'News 24', 'Bangladesh Pratidin']
  recall label0: 0.041 | recall label1: 0.967 | AUC: 0.669


In [6]:
# 11 — Cell 4: GroupDRO-lite (inverse-domain-frequency weighted LR), GroupKFold eval
def fit_scores_grouped(tr, te):
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=200000,
                        sublinear_tf=True, min_df=3)
    X = v.fit_transform(tr["text"])
    dom_counts = tr["domain"].value_counts()
    w = tr["domain"].map(lambda d: 1.0 / dom_counts[d]).values   # প্রতি domain সমান ওজন পায়
    clf = LogisticRegression(max_iter=300, class_weight="balanced", C=2.0)
    clf.fit(X, tr["label"], sample_weight=w)
    return clf.decision_function(v.transform(te["text"]))

for name, df in CORP.items():
    ys, ss = [], []
    for tr_i, te_i in GroupKFold(n_splits=5).split(df, groups=df["domain"]):
        tr, te = df.iloc[tr_i], df.iloc[te_i]
        ss.append(fit_scores_grouped(tr, te)); ys.append(te["label"].values)
    y, s = np.concatenate(ys), np.concatenate(ss); p = (s > 0).astype(int)
    print(name, "GroupDRO-lite pooled:",
          "bacc", round(balanced_accuracy_score(y, p), 3),
          "| recall0", round(((p==0)&(y==0)).sum()/(y==0).sum(), 3),
          "| recall1", round(((p==1)&(y==1)).sum()/(y==1).sum(), 3),
          "| AUC", round(roc_auc_score(y, s), 3))

c2020 GroupDRO-lite pooled: bacc 0.625 | recall0 0.276 | recall1 0.974 | AUC 0.891
c2025 GroupDRO-lite pooled: bacc 0.5 | recall0 0.002 | recall1 0.998 | AUC 0.597


In [7]:
# Cell 4b: capped domain reweighting
def fit_scores_capped(tr, te, cap=5.0):
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=200000,
                        sublinear_tf=True, min_df=3)
    X = v.fit_transform(tr["text"])
    dom_counts = tr["domain"].value_counts()
    raw_w = tr["domain"].map(lambda d: 1.0 / dom_counts[d]).values
    w = np.minimum(raw_w / raw_w.min(), cap)   # সবচেয়ে ছোট domain-এর weight-কে cap-এ সীমিত রাখা
    clf = LogisticRegression(max_iter=300, class_weight="balanced", C=2.0)
    clf.fit(X, tr["label"], sample_weight=w)
    return clf.decision_function(v.transform(te["text"]))

for name, df in CORP.items():
    ys, ss = [], []
    for tr_i, te_i in GroupKFold(n_splits=5).split(df, groups=df["domain"]):
        tr, te = df.iloc[tr_i], df.iloc[te_i]
        ss.append(fit_scores_capped(tr, te)); ys.append(te["label"].values)
    y, s = np.concatenate(ys), np.concatenate(ss); p = (s > 0).astype(int)
    print(name, "capped pooled:",
          "bacc", round(balanced_accuracy_score(y, p), 3),
          "| recall0", round(((p==0)&(y==0)).sum()/(y==0).sum(), 3),
          "| recall1", round(((p==1)&(y==1)).sum()/(y==1).sum(), 3),
          "| AUC", round(roc_auc_score(y, s), 3))

c2020 capped pooled: bacc 0.796 | recall0 0.599 | recall1 0.993 | AUC 0.973
c2025 capped pooled: bacc 0.505 | recall0 0.054 | recall1 0.956 | AUC 0.711
